# Fig S14 — CARMA flagship AD loci (data processing)

Reproduces the CARMA AD flagship examples with their original summary statistics, using the original UK Biobank LD reference and an ADSP reference: CS counts from SuSiE-RSS and SuSiE-rRSS (with SER fallback when the reliability flag is set), plus CARMA's own CS counts.

**Output:** `figure_data/FigS14.CARMA_Flagship.{rds,tsv,csv}` — loci with ≥ 1 SuSiE-RSS CS under the original UK Biobank reference. The Sankey flow counts in `Figures/Figure_6/FigS14.CARMA_Flagship.ipynb` come from this table.

In [1]:
source("../../R/utils.R")

`%||%` <- function(a, b) if (is.null(a) || length(a) == 0) b else a

## one row per region: CS counts + reliability diagnostics of the SuSiE-rRSS fit
summarize_results <- function(result_dir,
                              eb_name  = "fit_eb",
                              rss_name = "fit_rss") {
  files <- list.files(result_dir, pattern = "\\.rds$", full.names = TRUE)
  if (length(files) == 0) stop("No .rds files in ", result_dir)

  n_cs <- function(fit) if (is.null(fit[["sets"]][["cs"]])) 0L
                        else length(fit[["sets"]][["cs"]])

  rows <- lapply(files, function(f) {
    tryCatch({
      obj <- readRDS(f)

      # index: prefer stored field, else parse region_<NNN>_ from filename
      idx <- obj[["index"]]
      if (is.null(idx)) {
        m   <- regmatches(basename(f), regexpr("region_([0-9]+)", basename(f)))
        idx <- if (length(m)) as.integer(sub("region_", "", m)) else NA_integer_
      }

      eb  <- obj[[eb_name]]
      rss <- obj[[rss_name]]
      flag  <- eb[["R_finite_diagnostics"]][["R_reliability_flag"]]
      diag  <- eb[["R_finite_diagnostics"]]
      tibble::tibble(
        index            = idx,
        base             = obj[["base"]] %||% sub("\\.rds$", "", basename(f)),
        n_cs_eb          = n_cs(eb),
        n_cs_rss         = n_cs(rss),
        reliability_flag = if (is.null(flag)) NA_character_
                           else paste(flag, collapse = ";"),
        Q_art            = as.numeric(diag$Q_art %||% NA_real_),
        D_CS_max         = if (!is.null(diag$bf_attenuation$cs_max))
                             suppressWarnings(max(diag$bf_attenuation$cs_max, na.rm = TRUE))
                           else NA_real_,
        file             = basename(f)
      )
    }, error = function(e) {
      cat(sprintf("[FAIL] %s: %s\n", basename(f), conditionMessage(e)))
      NULL
    })
  })

  dplyr::bind_rows(rows) %>% dplyr::arrange(index)
}

In [2]:
ref_tb <- read_csv("/sc/arion/projects/ad-omics/haochen/rss_qc/CARMA/CARMA_2023_AD/ref_table.csv")

res_1kg  <- summarize_results("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/result_KG") %>%
  left_join(ref_tb, by = c("base" = "locus.name"))
res_ADSP <- summarize_results("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/CARMA_examples/result_ADSP/") %>%
  left_join(ref_tb, by = c("base" = "locus.name"))

Rows: 32 Columns: 12
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (5): Index_snp, ref, alt, locus.name, locus
dbl (6): chr, region_start, region_end, pos, Z, p.value
lgl (1): including

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [3]:
cs_1kg <- res_1kg %>%
  select(
    base,
    n_cs_eb_1kg = n_cs_eb,
    n_cs_rss_1kg = n_cs_rss,
    reliability_flag_1kg = reliability_flag,
    Q_art_1kg = Q_art,
    D_CS_max_1kg = D_CS_max,
    chr, region_start, region_end, locus, Index_snp, Z, p.value
  )

cs_ADSP <- res_ADSP %>%
  select(
    base,
    n_cs_eb_ADSP = n_cs_eb,
    n_cs_rss_ADSP = n_cs_rss,
    reliability_flag_ADSP = reliability_flag,
    Q_art_ADSP = Q_art,
    D_CS_max_ADSP = D_CS_max
  )

cs_summary <- cs_1kg %>%
  left_join(cs_ADSP, by = "base") %>%
  select(
    base, locus,
    n_cs_rss_1kg,  n_cs_eb_1kg,  reliability_flag_1kg, Q_art_1kg, D_CS_max_1kg,
    n_cs_rss_ADSP, n_cs_eb_ADSP, reliability_flag_ADSP, Q_art_ADSP, D_CS_max_ADSP
  ) %>%
  mutate(
    cs_final_SER_1kg  = if_else(as.logical(reliability_flag_1kg),  1L, n_cs_eb_1kg),
    cs_final_SER_ADSP = if_else(as.logical(reliability_flag_ADSP), 1L, n_cs_eb_ADSP)
  )

## CARMA's own results

In [4]:
carma_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/CARMA/result_CARMA"
files <- list.files(carma_dir, pattern = "\\.rds$", full.names = TRUE)

n_cs_CARMA_tb <- map_dfr(files, function(f) {
  res <- readRDS(f)
  tibble(base       = str_extract(basename(f), "p_chr[0-9XYxy]+_[0-9]+"),
         n_cs_CARMA = res$n_cs %||% NA_integer_)
})

cs_summary <- cs_summary %>%
  left_join(n_cs_CARMA_tb, by = "base")

In [5]:
flagship <- cs_summary %>% filter(n_cs_rss_1kg >= 1)
flagship

save_data(flagship, "FigS14.CARMA_Flagship")
write_tsv(flagship, file.path(data_dir, "FigS14.CARMA_Flagship.tsv"))

base,locus,n_cs_rss_1kg,n_cs_eb_1kg,reliability_flag_1kg,Q_art_1kg,D_CS_max_1kg,n_cs_rss_ADSP,n_cs_eb_ADSP,reliability_flag_ADSP,Q_art_ADSP,D_CS_max_ADSP,cs_final_SER_1kg,cs_final_SER_ADSP,n_cs_CARMA
<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<int>,<int>,<chr>,<dbl>,<dbl>,<int>,<int>,<int>
p_chr1_1,ADAMTS4,1,1,TRUE,0.19758728,4.263069e-04,1,1,TRUE,0.15876463,9.603625e-04,1,1,1
p_chr1_2,CR1,4,4,TRUE,0.09881011,1.638222e+03,1,1,FALSE,0.09039674,6.495130e-04,1,1,1
p_chr2_1,BIN1,9,9,TRUE,0.03794413,2.759102e+04,6,7,TRUE,0.03768019,2.145332e+04,1,1,1
p_chr3_1,HESX1,1,1,TRUE,0.12433925,3.243696e-03,1,1,FALSE,0.09667334,3.837425e-03,1,1,1
p_chr4_1,CLNK,6,1,TRUE,0.10723957,4.379723e-04,1,1,FALSE,0.07961758,2.917774e-04,1,1,1
p_chr4_2,HS3ST1,7,5,TRUE,0.07115139,1.467107e+04,5,5,TRUE,0.08891375,2.131058e+03,1,1,1
p_chr6_2,TREM2,2,1,FALSE,0.07531905,7.954265e-02,2,1,FALSE,0.04861680,7.863682e-02,1,1,1
p_chr6_3,CD2AP,6,8,TRUE,0.05128309,2.961898e+03,4,5,TRUE,0.06658757,8.832317e+03,1,1,1
p_chr7_1,ZCWPW1,10,3,TRUE,0.16511969,5.361028e+02,1,1,TRUE,0.15469440,4.968659e-03,1,1,1
